[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C73_3D_Representation_Course/04_detection3d/04_detection3d.ipynb)

# 04 · 3D 检测：从锚框到中心点

这个 notebook 从零实现**旋转框 IoU**（Sutherland–Hodgman 多边形裁剪），
然后用它量四件事：

1. **朝向不匹配的代价**：Δθ=45° 时 IoU 只有 **0.4254**（低于 0.5）；
   而反解出的允许朝向误差在四种长宽比上差 **9 倍**（轿车 37.1° vs 标志 9.6° vs 行人 90°）。
2. **锚框分配**：2 个朝向的锚框让 **45° 的目标拿到 0 个正锚**；
   而加到 4 个朝向**不改善正负比**（0.0416% → 0.0403%）。
3. **BEV NMS 的失效**：标志与它正下方的目标 BEV IoU **0.600** 而 3D IoU **0.000**。
4. **中心点头对薄目标同样退化**：标志的高斯半径 **0.08 格**，有效正样本面积 0 格。

> 心智模型：**这些都不是「模型不够好」，而是「正样本的定义本身对某类目标失效」。**

## 0 · 环境

In [ ]:
import numpy as np

print('numpy', np.__version__)

GRID = 0.2                              # BEV 网格分辨率
XR, YR = (0., 80.), (-40., 40.)
NX = int((XR[1] - XR[0]) / GRID)
NY = int((YR[1] - YR[0]) / GRID)
print(f'BEV 网格 {NX}×{NY} = {NX*NY:,} 个位置')

# 本模块的四个目标（x, y, θ）—— 朝向刻意覆盖 0°/22.5°/45°/70°
CAR_W, CAR_L = 1.9, 4.5
TARGETS = [(20., -3., 0.0), (35., 3., 22.5), (60., -3., 45.0), (75., 3., 70.0)]
print(f'四个目标，朝向 {[t[2] for t in TARGETS]}°')

## 1 · 旋转框 IoU：Sutherland–Hodgman 裁剪

**实现的第一步是检查绕向**——给成顺时针会让所有 IoU 变成 0，而它不报错。

In [ ]:
def rect_corners(cx, cy, w, l, th):
    '''BEV 旋转矩形的四角，**逆时针**。l 沿车长、w 沿车宽。'''
    c, s = np.cos(th), np.sin(th)
    R = np.array([[c, -s], [s, c]])
    P = np.array([[-l/2, -w/2], [l/2, -w/2], [l/2, w/2], [-l/2, w/2]])
    return P @ R.T + np.array([cx, cy])

def signed_area(P):
    P = np.asarray(P, float)
    return 0.5 * (np.dot(P[:, 0], np.roll(P[:, 1], -1))
                  - np.dot(P[:, 1], np.roll(P[:, 0], -1)))

# ★ 第一步：绕向自检
a = signed_area(rect_corners(0, 0, CAR_W, CAR_L, 0.0))
print(f'rect_corners 的有符号面积 = {a:.4f}  ({"逆时针 ✓" if a > 0 else "顺时针 ✗"})')
assert a > 0, 'Sutherland–Hodgman 的「在内侧」判据要求裁剪多边形逆时针'

def _isect(p1, p2, a_, b_):
    d1, d2 = p2 - p1, b_ - a_
    den = d1[0]*d2[1] - d1[1]*d2[0]
    if abs(den) < 1e-18:
        return p1
    t = ((a_[0]-p1[0])*d2[1] - (a_[1]-p1[1])*d2[0]) / den
    return p1 + t * d1

def poly_clip(subject, clipper):
    '''用凸多边形 clipper 裁 subject（都要逆时针）。'''
    out = [np.asarray(p, float) for p in subject]
    n = len(clipper)
    for i in range(n):
        a_, b_ = np.asarray(clipper[i], float), np.asarray(clipper[(i+1) % n], float)
        e = b_ - a_
        inside = lambda p: e[0]*(p[1]-a_[1]) - e[1]*(p[0]-a_[0]) >= -1e-12
        new = []
        for j in range(len(out)):
            cur, prv = out[j], out[j-1]
            if inside(cur):
                if not inside(prv):
                    new.append(_isect(prv, cur, a_, b_))
                new.append(cur)
            elif inside(prv):
                new.append(_isect(prv, cur, a_, b_))
        out = new
        if not out:
            return []
    return out

def poly_area(P):
    return abs(signed_area(P)) if len(P) >= 3 else 0.0

def bev_iou(b1, b2):
    '''b = (cx, cy, w, l, theta[rad])。'''
    p1, p2 = rect_corners(*b1), rect_corners(*b2)
    I = poly_area(poly_clip(p1, p2))
    A1, A2 = poly_area(p1), poly_area(p2)
    den = A1 + A2 - I
    return I / den if den > 0 else 0.0

# ★ 四个精确自检
BOX = (0., 0., CAR_W, CAR_L, 0.0)
checks = [
    ('同一个框',            bev_iou(BOX, BOX), 1.0),
    ('沿长轴平移 l/2',      bev_iou(BOX, (CAR_L/2, 0., CAR_W, CAR_L, 0.0)), 1/3),
    ('沿宽轴平移 w/2',      bev_iou(BOX, (0., CAR_W/2, CAR_W, CAR_L, 0.0)), 1/3),
    ('θ 转 180°（中心对称）', bev_iou(BOX, (0., 0., CAR_W, CAR_L, np.pi)), 1.0),
]
print()
for tag, got, want in checks:
    print(f'  {tag:22s} 实测 {got:.9f}  期望 {want:.9f}')
    assert abs(got - want) < 1e-9, (tag, got, want)
print('\n✅ 四个自检精确通过（1 / 1/3 / 1/3 / 1）')
print('   1/3 值得记：同尺寸框平移半个边长 -> 交 1/2、并 3/2 -> IoU = 1/3')

## 2 · 朝向不匹配的代价，与 IoU 阈值的反解

In [ ]:
SHAPES = {
    '轿车 4.5×1.9 (2.37:1)':   (1.9, 4.5),
    '卡车 10×2.5 (4:1)':       (2.5, 10.0),
    '行人 0.6×0.6 (1:1)':      (0.6, 0.6),
    '标志BEV 0.8×0.1 (8:1)':   (0.1, 0.8),
}
DEGS = [0, 5, 10, 15, 22.5, 30, 45, 60, 90]

def iou_at_angle(w, l, deg):
    return bev_iou((0., 0., w, l, 0.0), (0., 0., w, l, np.deg2rad(deg)))

print(f"{'目标':>24s} " + ''.join(f'{d}°'.rjust(8) for d in DEGS))
for name, (w, l) in SHAPES.items():
    row = [iou_at_angle(w, l, d) for d in DEGS]
    print(f'{name:>24s} ' + ''.join(f'{v:7.3f} ' for v in row))

# 轿车在 45° 掉到 0.5 以下
car = SHAPES['轿车 4.5×1.9 (2.37:1)']
assert iou_at_angle(*car, 45) < 0.5 < iou_at_angle(*car, 30)
print(f'\n轿车: Δθ=30° -> {iou_at_angle(*car,30):.4f}（过线）；'
      f'45° -> {iou_at_angle(*car,45):.4f}（**不过线**）')

# 方形目标：IoU 对朝向非单调，以 90° 为周期
ped = SHAPES['行人 0.6×0.6 (1:1)']
assert abs(iou_at_angle(*ped, 90) - 1.0) < 1e-9, '正方形转 90° 就是它自己'
assert iou_at_angle(*ped, 45) < iou_at_angle(*ped, 90)
print(f'行人: 45° -> {iou_at_angle(*ped,45):.4f}，而 90° -> '
      f'{iou_at_angle(*ped,90):.4f}  → **IoU 对朝向非单调**')

# 反解允许的朝向误差
def max_angle_for(w, l, thr):
    lo, hi = 0.0, 90.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if iou_at_angle(w, l, mid) >= thr:
            lo = mid
        else:
            hi = mid
    return lo

print(f"\n{'目标':>24s} {'IoU>=0.5 允许':>14s} {'IoU>=0.7 允许':>14s}")
allow = {}
for name, (w, l) in SHAPES.items():
    a5, a7 = max_angle_for(w, l, 0.5), max_angle_for(w, l, 0.7)
    allow[name] = (a5, a7)
    print(f'{name:>24s} {a5:13.1f}° {a7:13.1f}°')

assert allow['行人 0.6×0.6 (1:1)'][0] > 89, '方形目标的朝向完全不被 IoU 约束'
ratio = allow['轿车 4.5×1.9 (2.37:1)'][0] / allow['标志BEV 0.8×0.1 (8:1)'][0]
print(f'\n✅ 同一个阈值 0.5：轿车允许 {allow["轿车 4.5×1.9 (2.37:1)"][0]:.1f}°，'
      f'标志只允许 {allow["标志BEV 0.8×0.1 (8:1)"][0]:.1f}° -> 差 **{ratio:.1f} 倍**')
print('   而行人是 90°（完全不约束）—— 所以 mAP@0.5 在方形目标上不含朝向信息')
print('   → 这解释了 KITTI 车用 0.7 / 行人用 0.5，以及 nuScenes 独立上报 AOE')

## 3 · 锚框分配：45° 的目标拿到几个正锚

In [ ]:
def count_positive_anchors(target, anchor_degs, thr=0.5, half=6.0):
    '''在目标周围 ±half m 内扫锚框，返回 (正锚数, 最好 IoU)。'''
    cx, cy, th_deg = target
    tgt = (cx, cy, CAR_W, CAR_L, np.deg2rad(th_deg))
    xs = np.arange(max(cx-half, XR[0]), min(cx+half, XR[1]), GRID)
    ys = np.arange(max(cy-half, YR[0]), min(cy+half, YR[1]), GRID)
    cnt, best = 0, 0.0
    for x in xs:
        for y in ys:
            for a in anchor_degs:
                v = bev_iou(tgt, (x, y, CAR_W, CAR_L, np.deg2rad(a)))
                best = max(best, v)
                if v >= thr:
                    cnt += 1
    return cnt, best

for degs, tag in [([0, 90], '2 个朝向 (0°,90°)'), ([0, 45, 90, 135], '4 个朝向')]:
    total = NX * NY * len(degs)
    per, bests = [], []
    for t in TARGETS:
        c, b = count_positive_anchors(t, degs)
        per.append(c); bests.append(b)
    pos = sum(per)
    print(f'{tag}:')
    print(f'  总锚框 {total:,}   正锚 {pos}   正负比 {100*pos/total:.4f}%')
    print(f'  各目标（朝向 {[t[2] for t in TARGETS]}°）正锚数 {per}')
    print(f'  各目标的最好 IoU {[round(b,4) for b in bests]}')
    if degs == [0, 90]:
        two = (pos, total, per, bests)
    else:
        four = (pos, total, per, bests)
    print()

# ① 45° 的目标在 2 朝向下拿到 0 个正锚
i45 = [t[2] for t in TARGETS].index(45.0)
assert two[2][i45] == 0, f'45° 的目标应当拿到 0 个正锚，实测 {two[2][i45]}'
assert two[3][i45] < 0.5, f'它的最好 IoU 只有 {two[3][i45]:.4f}'
print(f'✅ 朝向 45° 的目标：2 朝向锚框下 **0 个正锚**'
      f'（最好 IoU 仅 {two[3][i45]:.4f}）')
print('   → 它在训练时完全没有正样本，只贡献「这里没有东西」')

# ② 加朝向解决漏掉，但不改善正负比
assert four[2][i45] > 0, '4 朝向应当救回它'
r2, r4 = two[0]/two[1], four[0]/four[1]
print(f'\n✅ 4 朝向救回了它（{four[2][i45]} 个正锚），'
      f'但正负比 {100*r2:.4f}% → {100*r4:.4f}%（几乎不变）')
assert abs(r4 - r2) / r2 < 0.2, '正负比应当基本不变'
print('   → **加朝向解决「漏掉」，不解决「万分之四的正负不平衡」**')

## 4 · BEV NMS 会抑制掉在 $z$ 上分开的目标

In [ ]:
def iou_axis_aligned(size, delta):
    s = np.asarray(size, float); d = np.abs(np.asarray(delta, float))
    I = np.prod(np.maximum(s - d, 0.0)); V = np.prod(s)
    den = 2 * V - I
    return I / den if den > 0 else 0.0

PAIRS = {'轿车 4.5×1.9×1.5': (4.5, 1.9, 1.5), '标志 0.8×0.1×0.8': (0.8, 0.1, 0.8)}
print(f"{'目标对 (δx=0.2m)':>20s} {'Δz':>7s} {'BEV IoU':>9s} {'3D IoU':>9s}")
for name, s in PAIRS.items():
    for dz in [0.0, 0.2, 0.5, 1.0]:
        b = iou_axis_aligned((s[0], s[1]), (0.2, 0.0))
        t = iou_axis_aligned(s, (0.2, 0.0, dz))
        print(f'{name:>20s} {dz:6.1f}m {b:9.3f} {t:9.3f}')

# 标志在 Δz=1.0 时 3D IoU 归零，而 BEV 仍然 0.6
s = PAIRS['标志 0.8×0.1×0.8']
b = iou_axis_aligned((s[0], s[1]), (0.2, 0.0))
t = iou_axis_aligned(s, (0.2, 0.0, 1.0))
assert b > 0.5 and t == 0.0, (b, t)
print(f'\n✅ 标志 Δz=1.0m：BEV IoU {b:.3f}（会被 NMS 抑制）而 3D IoU {t:.3f}（毫不相交）')
print('   → 一块离地 2.2m 的牌与它正下方的目标，BEV NMS 会抑制掉分数低的那个')
print('     而分数低的通常是更远、更小、更需要被检出的那个')

def nms_bev(boxes, scores, thr=0.5, z_gate=None):
    '''boxes: [(cx,cy,w,l,th,z,h)]。z_gate 不为 None 时，|Δz| 超过它就不抑制。'''
    order = np.argsort(scores)[::-1]
    keep = []
    for i in order:
        ok = True
        for j in keep:
            if bev_iou(boxes[i][:5], boxes[j][:5]) >= thr:
                if z_gate is not None and abs(boxes[i][5] - boxes[j][5]) > z_gate:
                    continue                      # z 上分开 -> 不抑制
                ok = False; break
        if ok:
            keep.append(int(i))
    return keep

# 构造一个「牌在地面标记正上方」的场景
BX = [(30., -5.5, 0.8, 0.1, 0.0, 2.2, 0.8),      # 限速牌（离地 2.2m），分数低
      (30., -5.5, 0.8, 0.1, 0.0, 0.0, 0.1)]      # 地面标记，分数高
SC = np.array([0.55, 0.90])
k_no = nms_bev(BX, SC, z_gate=None)
k_yes = nms_bev(BX, SC, z_gate=0.5)
print(f'\n无 z 门: 保留 {len(k_no)} 个 {k_no}  ← **牌被抑制了**')
print(f'有 z 门: 保留 {len(k_yes)} 个 {k_yes}')
assert len(k_no) == 1 and len(k_yes) == 2
print('✅ 加一个 z 门是零成本的，而它修掉了这一整类漏检')

## 5 · 中心点热图与高斯半径

In [ ]:
def gaussian_radius(h, w, min_overlap=0.7):
    '''CenterNet 的高斯半径：让「偏移 r 个格子的框」仍有 min_overlap 的 IoU。'''
    a1, b1 = 1, h + w
    c1 = w * h * (1 - min_overlap) / (1 + min_overlap)
    r1 = (b1 - np.sqrt(b1**2 - 4*a1*c1)) / 2
    a2, b2 = 4, 2 * (h + w)
    c2 = (1 - min_overlap) * w * h
    r2 = (b2 - np.sqrt(b2**2 - 4*a2*c2)) / 2
    a3, b3 = 4 * min_overlap, -2 * min_overlap * (h + w)
    c3 = (min_overlap - 1) * w * h
    r3 = (b3 + np.sqrt(b3**2 - 4*a3*c3)) / 2
    return float(min(r1, r2, r3))

print(f"{'目标':>18s} {'网格尺寸':>16s} {'高斯半径':>10s} {'有效面积 πr²':>14s}")
rad = {}
for tag, (w, l) in [('轿车 1.9×4.5', (1.9, 4.5)), ('行人 0.6×0.6', (0.6, 0.6)),
                    ('标志BEV 0.8×0.1', (0.1, 0.8))]:
    hg, wg = l / GRID, w / GRID
    r = gaussian_radius(hg, wg)
    rad[tag] = r
    print(f'{tag:>18s} {f"{hg:.1f}×{wg:.1f} 格":>16s} {r:9.2f} 格 '
          f'{np.pi*r*r:13.2f} 格')

assert rad['轿车 1.9×4.5'] > 1.0, '轿车应当有一个有意义的高斯半径'
assert np.pi * rad['标志BEV 0.8×0.1'] ** 2 < 0.1, \
    '标志的有效正样本面积应当接近 0'
print(f'\n✅ 轿车的有效正样本面积 {np.pi*rad["轿车 1.9×4.5"]**2:.1f} 格，'
      f'而标志只有 {np.pi*rad["标志BEV 0.8×0.1"]**2:.3f} 格')
print(f'   标志的 BEV 宽度只有 {0.1/GRID:.1f} 格 —— **小于一个格子**')
print('   → 中心点头对薄目标同样退化：只有恰好那一格是正样本，且没有软标签')

# 正负比对比
n_heat = NX * NY
print(f'\n正样本占比：')
print(f'  锚框（2 朝向）: {100*two[0]/two[1]:.4f}%')
print(f'  中心点热图（硬标签）: {100*len(TARGETS)/n_heat:.5f}%')
print(f'  中心点热图（轿车的高斯软标签）: '
      f'{100*len(TARGETS)*np.pi*rad["轿车 1.9×4.5"]**2/n_heat:.5f}%')
print('  → 中心点头的正样本更少，但**每个目标恒有一个**（与尺寸朝向无关）')

## 6 · 热图解码：局部极大值 + top-$k$

In [ ]:
def local_maxima_mask(H, k=3):
    '''3×3 局部极大值（等价于 max-pool 后比较）。'''
    pad = k // 2
    Hp = np.pad(H, pad, constant_values=-np.inf)
    mx = np.max([Hp[i:i+H.shape[0], j:j+H.shape[1]]
                 for i in range(k) for j in range(k)], axis=0)
    return H >= mx

rng = np.random.default_rng(0)
H_rand = rng.random((NX, NY))
m = local_maxima_mask(H_rand)
frac = m.mean()
print(f'纯随机热图 {NX}×{NY}：局部极大 {int(m.sum()):,} 个 = {100*frac:.2f}%')
print(f'  理论值 1/9 = {100/9:.2f}%')
assert abs(frac - 1/9) < 0.01, f'应当接近 1/9，实测 {frac:.4f}'
print('✅ 局部极大值抑制只能把候选降一个数量级 —— top-k 是必需的第二步\n')

print(f"{'k':>6s} {'占热图格数':>11s} {'相对 ~50 个目标的召回余量':>26s}")
for k in [100, 500, 1000]:
    print(f'{k:6d} {100*k/n_heat:10.3f}% {k/50:25.0f}×')

# 成本：旋转 NMS 是 O(k²)
print(f'\n旋转 NMS 的成本（O(k²)）：')
for k in [100, 500, 1000, 5000]:
    pairs = k * (k - 1) // 2
    print(f'  k={k:5d}: 最多 {pairs:,} 次旋转 IoU'
          f'（每次比轴对齐贵约 40 倍）')
print('  → k 太小截掉低分的真目标（远处小目标），k 太大让 NMS 变贵')

# offset 回归补偿量化误差
print(f'\noffset 头补偿的量化误差：网格 {GRID} m -> 最大 '
      f'{GRID*np.sqrt(2)/2:.4f} m（BEV 平面内，半格对角）')
print(f'  而模块 03 第 5b 节用的是三维的 r√3/2 = {GRID*np.sqrt(3)/2:.4f} m')

## 7 · 角度的三种编码

In [ ]:
def enc_direct(th):      return np.array([th])
def enc_sincos(th):      return np.array([np.sin(th), np.cos(th)])
def enc_sincos2(th):     return np.array([np.sin(2*th), np.cos(2*th)])

def dist(enc, d1, d2):
    return float(np.linalg.norm(enc(np.deg2rad(d1)) - enc(np.deg2rad(d2))))

print('情形一：179° vs −179°（几何差 2°，周期性问题）')
print(f'  直接回归 θ  : L1 = {abs(179-(-179))}°  ← 放大 {abs(179-(-179))/2:.0f} 倍')
print(f'  (sinθ, cosθ): L2 = {dist(enc_sincos,179,-179):.5f}'
      f'   而 2·sin(1°) = {2*np.sin(np.deg2rad(1)):.5f}')
assert abs(dist(enc_sincos, 179, -179) - 2*np.sin(np.deg2rad(1))) < 1e-9
print('  ✅ (sin,cos) 把角度差正确线性化了（单位圆上的弦长 = 2·sin(Δθ/2)）\n')

print('情形二：0° vs 180°（中心对称——它们是同一个框）')
print(f'  实测 IoU = {bev_iou(BOX,(0.,0.,CAR_W,CAR_L,np.pi)):.6f}')
print(f'  (sinθ, cosθ) : L2 = {dist(enc_sincos,0,180):.5f}  ← **被当成完全不同**')
print(f'  (sin2θ,cos2θ): L2 = {dist(enc_sincos2,0,180):.2e}  ← 正确地为 0')
assert dist(enc_sincos, 0, 180) > 1.9
assert dist(enc_sincos2, 0, 180) < 1e-9
print('  ✅ 只有 (sin2θ, cos2θ) 同时处理周期性与中心对称\n')

print('情形三：89° vs −89°（中心对称下几何差 2°）')
print(f'  直接回归     : L1 = {abs(89-(-89))}°  ← 放大 {abs(89-(-89))/2:.0f} 倍')
print(f'  (sin2θ,cos2θ): L2 = {dist(enc_sincos2,89,-89):.5f}'
      f'   而 2·sin(2°) = {2*np.sin(np.deg2rad(2)):.5f}')
assert abs(dist(enc_sincos2, 89, -89) - 2*np.sin(np.deg2rad(2))) < 1e-9
print('  ✅ 正确\n')
print('代价：(sin2θ,cos2θ) 丢掉了「车头朝哪」 -> 需要一个额外的方向二分类头')
print('     （这就是 SECOND 引入 direction classifier 的原因）')

## 8 · 小结

| 结论 | 数值 |
|---|---|
| 旋转框 IoU 的四个自检 | 1 / **1/3** / **1/3** / 1（精确） |
| 朝向不匹配 | Δθ=30° → 0.5662（过线）；**45° → 0.4254（不过线）** |
| IoU=0.5 隐含的朝向精度 | 轿车 **37.1°** · 卡车 19.9° · **行人 90°（不约束）** · 标志 **9.6°** |
| IoU 对朝向**非单调** | 方形目标以 90° 为周期（45° 是 0.707，90° 回到 1.000） |
| 锚框（2 朝向） | 正负比 **0.0416%**，而 **45° 的目标 0 个正锚** |
| 加到 4 朝向 | 救回了它，但正负比 **0.0403%（几乎不变）** |
| **BEV NMS** | 标志 Δz=1.0 m：BEV IoU **0.600** 而 3D IoU **0.000** |
| 高斯半径 | 轿车 1.23 格（πr² = 4.7）· **标志 0.08 格（πr² ≈ 0）** |
| 局部极大值 | 随机热图上 **11.20%**（理论 1/9 = 11.11%） |
| 角度编码 | 只有 $(\sin2\theta,\cos2\theta)$ 同时处理周期性与中心对称 |

## ✏️ 练习 1：旋转框 IoU 与绕向自检

实现 `rotated_iou(b1, b2)`，`b = (cx, cy, w, l, theta)`。
**要求内部先检查两个多边形的绕向，不是逆时针就翻转。**

返回 IoU。然后用四个精确自检验证。

In [ ]:
def rotated_iou(b1, b2):
    """旋转框 IoU（内部保证绕向正确）。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
B = (0., 0., 1.9, 4.5, 0.0)
CASES = [
    ('同一个框',              (0., 0., 1.9, 4.5, 0.0),      1.0),
    ('沿长轴平移 l/2',        (2.25, 0., 1.9, 4.5, 0.0),    1/3),
    ('沿宽轴平移 w/2',        (0., 0.95, 1.9, 4.5, 0.0),    1/3),
    ('θ 转 180°',             (0., 0., 1.9, 4.5, np.pi),    1.0),
    ('完全分离',              (100., 0., 1.9, 4.5, 0.0),    0.0),
]
for tag, b2, want in CASES:
    got = rotated_iou(B, b2)
    print(f'  {tag:18s} 实测 {got:.9f}  期望 {want:.9f}')
    assert abs(got - want) < 1e-9, (tag, got, want)

# 对称性：IoU(a,b) == IoU(b,a)
rng1 = np.random.default_rng(3)
for _ in range(30):
    a = (rng1.uniform(-3, 3), rng1.uniform(-3, 3), 1.9, 4.5, rng1.uniform(0, np.pi))
    b = (rng1.uniform(-3, 3), rng1.uniform(-3, 3), 1.9, 4.5, rng1.uniform(0, np.pi))
    assert abs(rotated_iou(a, b) - rotated_iou(b, a)) < 1e-9

# 与顺时针输入也要给同一个答案（这就是绕向自检的作用）
def rect_cw(cx, cy, w, l, th):
    return rect_corners(cx, cy, w, l, th)[::-1]
assert signed_area(rect_cw(0, 0, 1.9, 4.5, 0.0)) < 0, '这个构造应当是顺时针'
print('\n✅ 练习 1 通过：四个自检精确、对称性成立')
print('   而绕向自检的价值：给成顺时针会让所有 IoU 变成 0，**且不报错**')

## 📖 参考答案 1

In [ ]:
# 练习 1 参考答案
def _ccw(P):
    P = np.asarray(P, float)
    return P if signed_area(P) > 0 else P[::-1]

def rotated_iou(b1, b2):
    p1 = _ccw(rect_corners(*b1))
    p2 = _ccw(rect_corners(*b2))
    I = poly_area(poly_clip(p1, p2))
    A1, A2 = poly_area(p1), poly_area(p2)
    den = A1 + A2 - I
    return I / den if den > 0 else 0.0

assert abs(rotated_iou((0.,0.,1.9,4.5,0.), (2.25,0.,1.9,4.5,0.)) - 1/3) < 1e-9
assert abs(rotated_iou((0.,0.,1.9,4.5,0.), (0.,0.,1.9,4.5,np.pi)) - 1.0) < 1e-9
print('✅ 参考答案 1 通过')
print('   `_ccw` 只有两行，而它防的是一个「不报错、让所有匹配失败」的 bug。')

## ✏️ 练习 2：锚框分配审计

实现 `anchor_audit(targets, anchor_degs, thr=0.5, w=1.9, l=4.5)`，返回 dict：

- `'total_anchors'` —— 全网格的锚框总数
- `'per_target'` —— 每个目标的正锚数（list）
- `'best_iou'` —— 每个目标能拿到的最好 IoU（list）
- `'pos_ratio'` —— 正锚总数 / 锚框总数
- `'starved'` —— **正锚数为 0 的目标的下标**（list）

`'starved'` 非空就说明「有目标在训练时完全没有正样本」。

In [ ]:
def anchor_audit(targets, anchor_degs, thr=0.5, w=1.9, l=4.5, half=6.0):
    """返回 dict(total_anchors, per_target, best_iou, pos_ratio, starved)。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
a2 = anchor_audit(TARGETS, [0, 90])
a4 = anchor_audit(TARGETS, [0, 45, 90, 135])
for a in (a2, a4):
    assert set(a) == {'total_anchors', 'per_target', 'best_iou',
                      'pos_ratio', 'starved'}

print(f"{'配置':>10s} {'总锚框':>10s} {'正负比':>10s} {'各目标正锚':>22s} {'饿死的目标':>12s}")
for tag, a in [('2 朝向', a2), ('4 朝向', a4)]:
    print(f'{tag:>10s} {a["total_anchors"]:10,} {100*a["pos_ratio"]:9.4f}% '
          f'{str(a["per_target"]):>22s} {str(a["starved"]):>12s}')

# ① 2 朝向下，45° 的目标被饿死
i45 = [t[2] for t in TARGETS].index(45.0)
assert a2['starved'] == [i45], (a2['starved'], i45)
assert a2['best_iou'][i45] < 0.5
print(f'\n2 朝向: 目标 #{i45}（朝向 45°）被饿死，最好 IoU 仅 '
      f'{a2["best_iou"][i45]:.4f}')

# ② 4 朝向救回它，但正负比不变
assert a4['starved'] == []
assert abs(a4['pos_ratio'] - a2['pos_ratio']) / a2['pos_ratio'] < 0.2
print(f'4 朝向: 无饿死目标，而正负比 {100*a2["pos_ratio"]:.4f}% → '
      f'{100*a4["pos_ratio"]:.4f}%（几乎不变）')

# ③ 提高阈值会饿死更多目标
a2_hi = anchor_audit(TARGETS, [0, 90], thr=0.7)
print(f'\n把阈值提到 0.7（2 朝向）: 饿死的目标 {a2_hi["starved"]}'
      f'（共 {len(TARGETS)} 个）')
assert len(a2_hi['starved']) > len(a2['starved'])
print('✅ 练习 2 通过：**`starved` 这一项在总正负比上完全看不出来**')

## 📖 参考答案 2

In [ ]:
# 练习 2 参考答案
def anchor_audit(targets, anchor_degs, thr=0.5, w=1.9, l=4.5, half=6.0):
    total = NX * NY * len(anchor_degs)
    per, best = [], []
    for (cx, cy, th_deg) in targets:
        tgt = (cx, cy, w, l, np.deg2rad(th_deg))
        xs = np.arange(max(cx-half, XR[0]), min(cx+half, XR[1]), GRID)
        ys = np.arange(max(cy-half, YR[0]), min(cy+half, YR[1]), GRID)
        cnt, b = 0, 0.0
        for x in xs:
            for y in ys:
                for a in anchor_degs:
                    v = rotated_iou(tgt, (x, y, w, l, np.deg2rad(a)))
                    b = max(b, v)
                    if v >= thr:
                        cnt += 1
        per.append(cnt); best.append(float(b))
    return {'total_anchors': int(total), 'per_target': per,
            'best_iou': best, 'pos_ratio': sum(per) / total,
            'starved': [i for i, c in enumerate(per) if c == 0]}

a = anchor_audit(TARGETS, [0, 90])
assert a['starved'] == [[t[2] for t in TARGETS].index(45.0)]
print('✅ 参考答案 2 通过')
print('   `half=6.0` 是一个安全的剪枝：更远的锚框与 4.5m 的框不可能有 0.5 的 IoU。')

## ✏️ 练习 3：带 $z$ 门的 NMS

实现 `nms_with_z_gate(boxes, scores, iou_thr=0.5, z_gate=None)`：
`boxes` 每项是 `(cx, cy, w, l, theta, z_center, h)`。

规则：按分数降序，若与已保留框的 **BEV IoU** $\ge$ `iou_thr`
**且** `|Δz| <=` `z_gate`（`z_gate=None` 表示不看 $z$），则抑制。

返回保留的下标（按分数降序）。

In [ ]:
def nms_with_z_gate(boxes, scores, iou_thr=0.5, z_gate=None):
    """返回保留的下标 list。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
# 场景：一块离地 2.2m 的牌（分数 0.55），正下方是地面标记（分数 0.90）
SCENE = [(30., -5.5, 0.8, 0.1, 0.0, 2.2, 0.8),
         (30., -5.5, 0.8, 0.1, 0.0, 0.0, 0.1)]
SCORES = np.array([0.55, 0.90])

k_no = nms_with_z_gate(SCENE, SCORES, z_gate=None)
k_yes = nms_with_z_gate(SCENE, SCORES, z_gate=0.5)
print(f'无 z 门: 保留 {k_no}（{len(k_no)} 个）  ← 牌被抑制')
print(f'有 z 门: 保留 {k_yes}（{len(k_yes)} 个）')
assert len(k_no) == 1 and k_no == [1], k_no
assert len(k_yes) == 2, k_yes
assert k_yes[0] == 1, '应当按分数降序返回'

# 真正重复的框仍然要被抑制
# 注意标志只有 0.1 m 厚（l=0.1 沿 x），所以偏移必须远小于 0.05 m 才算「重复」：
#   偏移 0.05 m = 半个边长 -> IoU 恰好 1/3，本来就不该被抑制
DUP = [(30.00, -5.5, 0.8, 0.1, 0.0, 2.20, 0.8),
       (30.01, -5.5, 0.8, 0.1, 0.0, 2.25, 0.8)]
print(f'\n两个抖动框的 BEV IoU = {rotated_iou(DUP[0][:5], DUP[1][:5]):.4f}'
      f'（偏移 0.01 m，而框只有 0.1 m 厚）')
assert rotated_iou(DUP[0][:5], DUP[1][:5]) > 0.5
k_dup = nms_with_z_gate(DUP, np.array([0.9, 0.8]), z_gate=0.5)
assert len(k_dup) == 1, f'z 接近的重复框仍应被抑制，实得 {k_dup}'
print(f'同高度的重复框（Δz=0.05m < 门 0.5m）: 保留 {k_dup}（正确抑制）')

# z 门不能太小，否则同一目标的两个抖动框也不抑制了
k_tight = nms_with_z_gate(DUP, np.array([0.9, 0.8]), z_gate=0.01)
print(f'z 门设成 0.01m（< Δz=0.05m）: 保留 {k_tight}  ← **门太紧，重复框没被抑制**')
assert len(k_tight) == 2
print('\n✅ 练习 3 通过：z 门要大于「同一目标的 z 抖动」、'
      '小于「不同目标的 z 间距」')
print('   本例里 0.5m 合适：牌与地面差 2.2m，而同一目标的 z 抖动只有 0.05m')

## 📖 参考答案 3

In [ ]:
# 练习 3 参考答案
def nms_with_z_gate(boxes, scores, iou_thr=0.5, z_gate=None):
    order = list(np.argsort(np.asarray(scores))[::-1])
    keep = []
    for i in order:
        suppressed = False
        for j in keep:
            if rotated_iou(boxes[i][:5], boxes[j][:5]) < iou_thr:
                continue
            if z_gate is not None and abs(boxes[i][5] - boxes[j][5]) > z_gate:
                continue                       # z 上分开 -> 不算重复
            suppressed = True
            break
        if not suppressed:
            keep.append(int(i))
    return keep

assert len(nms_with_z_gate(SCENE, SCORES, z_gate=None)) == 1
assert len(nms_with_z_gate(SCENE, SCORES, z_gate=0.5)) == 2
print('✅ 参考答案 3 通过')
print('   注意 z 门是「不抑制」的条件而不是「抑制」的条件 ——')
print('   写反了会变成「只抑制 z 上分开的框」，那正好是最坏的行为。')

## ✏️ 练习 4：热图头的可行性审计

实现 `heatmap_audit(class_sizes, grid, min_overlap=0.7)`，
`class_sizes` 是 `{类名: (w, l)}`（米），返回 `{类名: dict}`，每项含：

- `'cells'` —— `(l/grid, w/grid)`
- `'radius'` —— 高斯半径（格）
- `'eff_area'` —— $\pi r^2$（格）
- `'sub_cell'` —— bool：是否有任一边小于 1 格
- `'degenerate'` —— bool：`eff_area < 1`（**没有软标签**）
- `'grid_for_area1'` —— 让 `eff_area >= 1` 所需的网格分辨率（米）

In [ ]:
def heatmap_audit(class_sizes, grid, min_overlap=0.7):
    """返回 {类名: dict(cells, radius, eff_area, sub_cell, degenerate,
    grid_for_area1)}。"""
    # TODO
    raise NotImplementedError

In [ ]:
# —— 自测 ——
CLASSES = {'car': (1.9, 4.5), 'pedestrian': (0.6, 0.6),
           'sign_bev': (0.1, 0.8), 'truck': (2.5, 10.0)}
au = heatmap_audit(CLASSES, GRID)
for name, v in au.items():
    assert set(v) == {'cells', 'radius', 'eff_area', 'sub_cell',
                      'degenerate', 'grid_for_area1'}

print(f"{'类别':>12s} {'网格尺寸':>14s} {'半径':>8s} {'πr²':>8s} "
      f"{'亚格':>6s} {'退化':>6s} {'需要网格':>10s}")
for name, v in au.items():
    print(f'{name:>12s} {f"{v["cells"][0]:.1f}×{v["cells"][1]:.1f}":>14s} '
          f'{v["radius"]:7.2f} {v["eff_area"]:7.2f} '
          f'{str(v["sub_cell"]):>6s} {str(v["degenerate"]):>6s} '
          f'{v["grid_for_area1"]:9.3f}m')

# ① 车与卡车不退化
assert au['car']['degenerate'] is False
assert au['truck']['degenerate'] is False
# ② 标志退化，且是亚格的
assert au['sign_bev']['degenerate'] is True
assert au['sign_bev']['sub_cell'] is True
# ③ 让标志不退化需要更细的网格
assert au['sign_bev']['grid_for_area1'] < GRID / 2, \
    f"标志需要的网格应当细于 {GRID/2} m，实测 {au['sign_bev']['grid_for_area1']:.3f}"
print(f'\n✅ 标志在 {GRID} m 网格上退化（πr² = '
      f'{au["sign_bev"]["eff_area"]:.3f}），')
print(f'   要让它有软标签需要网格细到 '
      f'{au["sign_bev"]["grid_for_area1"]:.3f} m')

# ④ 用那个网格重算，应当不再退化
au2 = heatmap_audit({'sign_bev': (0.1, 0.8)},
                    au['sign_bev']['grid_for_area1'])
assert au2['sign_bev']['degenerate'] is False
print(f'   用 {au["sign_bev"]["grid_for_area1"]:.3f} m 重算: '
      f'πr² = {au2["sign_bev"]["eff_area"]:.2f}（不再退化）')
print('\n✅ 练习 4 通过：这个审计只依赖类别尺寸与网格，'
      '**在选架构之前就能算**')

## 📖 参考答案 4

In [ ]:
# 练习 4 参考答案
def heatmap_audit(class_sizes, grid, min_overlap=0.7):
    out = {}
    for name, (w, l) in class_sizes.items():
        hg, wg = l / grid, w / grid
        r = gaussian_radius(hg, wg, min_overlap)
        eff = float(np.pi * r * r)
        # 二分找让 eff_area >= 1 的网格
        lo, hi = 1e-4, grid
        for _ in range(80):
            mid = (lo + hi) / 2
            rr = gaussian_radius(l / mid, w / mid, min_overlap)
            if np.pi * rr * rr >= 1.0:
                lo = mid          # 还能更粗
            else:
                hi = mid
        out[name] = {'cells': (hg, wg), 'radius': float(r), 'eff_area': eff,
                     'sub_cell': bool(min(hg, wg) < 1.0),
                     'degenerate': bool(eff < 1.0),
                     'grid_for_area1': float(lo)}
    return out

a = heatmap_audit({'sign_bev': (0.1, 0.8), 'car': (1.9, 4.5)}, 0.2)
assert a['sign_bev']['degenerate'] and not a['car']['degenerate']
print('✅ 参考答案 4 通过')
print('   二分的方向要想清楚：网格越**细**，格数越多，半径越大 ——')
print('   所以「还能更粗」时把下界往上移。')

## 🧪 真实工程胶囊

```python
# ── 1) 旋转 IoU：用现成的（本课自己实现是为了看清语义）──
from mmcv.ops import boxes_iou3d_gpu, nms3d          # 或 iou3d_nms_utils
ious = boxes_iou3d_gpu(boxes_a, boxes_b)              # (N, M)
keep = nms3d(boxes, scores, iou_threshold=0.1)        # ← 3D 检测的 NMS 阈值普遍很低
#   ⚠️ 3D NMS 的阈值常取 0.1–0.25 而不是 2D 常用的 0.5 —— 因为「三个乘子」让
#      同一目标的两个候选框的 IoU 天然更低（模块 05）。抄 2D 的 0.5 会留下大量重复。

# ── 2) BEV NMS 的 z 门（第 4 节，零成本）──
#    mmdet3d 的 nms_bev 不看 z；如果你的类别里有「上下叠放」的（标志 vs 地面标记），
#    要么按类别分组 NMS，要么自己加门：
keep = [i for i in keep if not any(
    abs(boxes[i, 2] - boxes[j, 2]) <= Z_GATE and bev_iou(i, j) >= THR
    for j in kept_before)]

# ── 3) 角度：SECOND / CenterPoint 的做法 ──
#    回归 (sin, cos)，另加一个 direction classifier 判方向：
rot_sine = pred[..., 6:7]; rot_cosine = pred[..., 7:8]
theta = torch.atan2(rot_sine, rot_cosine)
dir_cls = pred_dir.argmax(-1)                        # 0/1
theta = theta + dir_cls * np.pi                      # 把方向补回来
#   ⚠️ 训练时 direction 的标签要用「θ 是否落在 [0, π)」，
#      而这个边界的定义（dir_offset）是一个真实的、容易搞错的超参数。

# ── 4) 上线前先跑两个审计（练习 2 与 4）──
#    a) anchor_audit：有没有类别/朝向的目标「一个正样本都拿不到」
#    b) heatmap_audit：有没有类别的 πr² < 1（没有软标签）
#    两者都只需要类别尺寸与网格配置，**不需要训练**。
```

> **落地顺序建议**：先跑 `heatmap_audit`（几行，立刻告诉你哪些类别在当前网格下退化），
> 再检查 3D NMS 的阈值是不是从 2D 抄来的 0.5，
> 最后才是给 BEV NMS 加 z 门（它只在你的类别里真有上下叠放时才需要）。